# ETH/BTC Suspicious Pattern Analysis — interactive notebook

Notebook companion to `analyze.py`. Reproduces every finding in `REPORT.md`
with intermediate dataframes visible. Run cells top-to-bottom.

**Author:** Max Gorbuk · gorbuk@stanford.edu

**Submission:** DN Institute Market Data Challenge.

In [ ]:
import sys
from pathlib import Path
if Path('../src').exists():
    sys.path.insert(0, '..')
elif Path('src').exists():
    sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data import load_trades, load_orderbooks, summarize
from src.detector_imbalance import detect_buysell_imbalance, summarize_d1
from src.detector_signatures import detect_size_signatures, summarize_d2
from src.detector_pumpdump import detect_pumpdump, summarize_d3
from src.detector_liquidity import detect_liquidity_quality
from src.detector_bursts import detect_bursts, detect_time_of_day, detect_anchor_prices, summarize_d5

## 1. Load data

In [ ]:
def _find(name):
    for cand in [f'data/{name}', f'../data/{name}', f'../{name}']:
        if Path(cand).exists():
            return cand
    return f'data/{name}'  # last-resort default
TRADES = _find('eth-btc-trades.csv')
ORDERBOOKS = _find('eth-btc-orderbooks.csv')

trades = load_trades(TRADES)
ob = load_orderbooks(ORDERBOOKS)
summarize(trades, ob)

## 2. EDA — confirm the bimodality finding

In [ ]:
# Side-conditional size distribution. BUY median ~188 ETH, SELL median ~0.0014 ETH.
trades.groupby('side')['size'].describe()

In [ ]:
# Side semantics: aggressor or passive? — Compare trade price to contemporaneous mid.
ob_sorted = ob.sort_values('timestamp').reset_index(drop=True)
merged = pd.merge_asof(
    trades[['timestamp','side','price']].sort_values('timestamp'),
    ob_sorted[['timestamp','mid']],
    on='timestamp', direction='backward'
).dropna(subset=['mid'])
merged['price_vs_mid_bps'] = (merged['price'] - merged['mid']) / merged['mid'] * 1e4
merged.groupby('side')['price_vs_mid_bps'].describe()
# BUY > 0 (above mid → paid the ask) and SELL < 0 (below mid → hit the bid) confirms aggressor semantics.

## 3. Detector 1 — Imbalance

In [ ]:
d1 = detect_buysell_imbalance(trades)
summarize_d1(d1)

In [ ]:
# Flagged buckets — all NEGATIVE z (lulls in buy pressure, not spikes)
d1[d1['flag_any'].fillna(False)][['buy_count','sell_count','log_count_ratio','z_count']]

## 4. Detector 2 — Recurring sizes

In [ ]:
d2 = detect_size_signatures(trades)
for side in ('buy','sell'):
    print(f'\n--- {side.upper()} (n={d2[side]["n"]}, threshold={d2[side]["threshold"]}) ---')
    print(d2[side]['flagged'].head(10))

In [ ]:
# The 0.00026058 burst structure
target = 0.00026058
matches = trades[trades['size'].round(8) == target].sort_values('timestamp')
matches[['timestamp','side','price','size']]

In [ ]:
# Doubling-ladder Monte Carlo
buy_sizes = trades[trades.side=='buy']['size'].round(8).values
flagged_18 = d2['buy']['flagged']['size'].values

def count_2x(sizes, tol=1e-7):
    s_set = set(round(float(x), 8) for x in sizes)
    seen, pairs = set(), 0
    for x in s_set:
        y = round(2*x, 8)
        if y in s_set and tuple(sorted([x,y])) not in seen:
            pairs += 1
            seen.add(tuple(sorted([x,y])))
    return pairs

obs = count_2x(flagged_18)
rng = np.random.default_rng(42)
null = [count_2x(rng.choice(buy_sizes, size=18, replace=False)) for _ in range(2000)]
print(f'observed: {obs}, null mean: {np.mean(null):.2f}, P(null≥obs): {(np.array(null)>=obs).mean():.4f}')

## 5. Detector 3 — Pump-and-dump

In [ ]:
d3 = detect_pumpdump(trades)
print(f'candidates: {len(d3)}')
d3

## 6. Detector 4 — Liquidity / outside-spread

In [ ]:
d4 = detect_liquidity_quality(trades, ob)
d4['summary']

In [ ]:
# Outside-spread sells dominate; cluster at 09-01 20:38–20:42
out = d4['trades_outside']
print(f'side breakdown: {out["side"].value_counts().to_dict()}')
out[['timestamp','side','price','bid_price','ask_price']].head(15)

## 7. Detector 5 — Bursts / TOD / Anchors

In [ ]:
bursts = detect_bursts(trades, min_trades_per_second=5)
bursts

In [ ]:
tod = detect_time_of_day(trades)
tod

In [ ]:
anchors = detect_anchor_prices(trades, top_n=15)
anchors

## 8. Render all figures (calls into analyze.py logic)

In [ ]:
from pathlib import Path
fig_dir = Path('../figures' if Path('../figures').exists() else 'figures')
fig_dir.mkdir(exist_ok=True)

from src.plotting import (plot_imbalance, plot_signatures, plot_pumpdump,
                          plot_liquidity, plot_cooccurrence_v2, plot_bursts_and_tod)

plot_imbalance(d1, fig_dir / 'd1_imbalance.png')
plot_signatures(trades, d2, fig_dir / 'd2_signatures.png')
plot_pumpdump(trades, d3, fig_dir / 'd3_pumpdump.png')
plot_liquidity(d4, fig_dir / 'd4_liquidity.png')
plot_bursts_and_tod(bursts, tod, anchors, trades, fig_dir / 'd5_bursts_tod_anchors.png')
plot_cooccurrence_v2(d1, d2, d3, d4, trades, fig_dir / 'co_occurrence_timeline.png')
print('all figures written to', fig_dir)